In [41]:
import re
import fitz

class PDFParser:
    def __init__(self, pdf_path):
        self.pdf_path = pdf_path
        self.doc = fitz.open(pdf_path)

    def extract(self):
        """
        Returns:
        [
            {
                "page": int,
                "text": str
            }
        ]
        """
        parsed_pages = []

        for page_num, page in enumerate(self.doc, start=1):
            # Use 'blocks' instead of 'text' to preserve 
            # geometric paragraphs and row logic in tables
            blocks = page.get_text("blocks")
            
            # b[4] contains the actual text of the block
            # filter out empty blocks or images
            page_text = "\n".join([b[4] for b in blocks if isinstance(b[4], str)])

            parsed_pages.append({
                "page": page_num,
                "text": self._clean_text(page_text)
            })

        return parsed_pages

    def _clean_text(self, text):
        # Keep structure (important for section detection)
        text = re.sub(r'[ \t]+', ' ', text)
        text = re.sub(r'\n{2,}', '\n', text)
        return text.strip()


In [42]:
import uuid
import re

class Chunker:
    def __init__(self, max_tokens=500, overlap=80):
        self.max_tokens = max_tokens
        self.overlap = overlap

    def chunk(self, parsed_pages):
        """
        Main chunking pipeline
        """
        chunks = []

        current_chapter = None
        current_section = None

        buffer = ""
        buffer_pages = set()

        for page in parsed_pages:
            text = page["text"]
            page_num = page["page"]

            # Detect structure
            chapter = self._detect_chapter(text)
            section = self._detect_section(text)

            if chapter:
                current_chapter = chapter
            if section:
                current_section = section

            buffer += " " + text
            buffer_pages.add(page_num)

            # Token-based split
            while self._token_len(buffer) > self.max_tokens:
                words = buffer.split(" ")
                chunk_text = " ".join(words[:self.max_tokens])

                chunks.append(
                    self._create_chunk(
                        chunk_text,
                        current_chapter,
                        current_section,
                        list(buffer_pages)
                    )
                )

                leftover_words = words[self.max_tokens - self.overlap:]
                buffer = " ".join(leftover_words)
                buffer_pages = {page_num}

        # last chunk
        if buffer.strip():
            chunks.append(
                self._create_chunk(
                    buffer,
                    current_chapter,
                    current_section,
                    list(buffer_pages)
                )
            )

        return chunks

    # ---------- Helpers ---------- #

    def _detect_chapter(self, text):
        lines = text.split("\n")
        for i, line in enumerate(lines[:10]):
            line_clean = line.strip().lower()
            match = re.match(r'^chapter\s+(\d+)$', line_clean)
            if match:
                return {
                    "number": int(match.group(1)),
                    "title": None
                }

            if line_clean == "chapter":
                if i + 1 < len(lines):
                    next_line = lines[i + 1].strip()
                    if next_line.isdigit():
                        return {
                            "number": int(next_line),
                            "title": None
                        }

        return None

    def _detect_section(self, text):
        lines = text.split("\n")

        for line in lines:
            line = line.strip()

            # must start line, avoid decimals like 0.1 mm
            match = re.match(r'^(\d+\.\d+)\s+([A-Za-z][A-Za-z\s\-\—]+)', line)
            if match:
                return {
                    "number": match.group(1),
                    "title": match.group(2).strip()
                }

        return None

    def _token_len(self, text):
        # Update: Split only by spaces to preserve newlines
        return len(text.split(" "))

    def _create_chunk(self, text, chapter, section, pages):
        return {
            "id": str(uuid.uuid4()),
            "text": text.strip(),
            "metadata": {
                "chapter": chapter,
                "section": section,
                "page_range": [min(pages), max(pages)] if pages else []
            }
        }


In [43]:
import os
import glob
import json
import pandas as pd
import fitz

# --- Locate data folder ---
current = os.getcwd()
while not os.path.exists(os.path.join(current, "data")):
    parent = os.path.dirname(current)
    if parent == current:
        raise Exception("Could not find 'data' directory.")
    current = parent

data_path = os.path.join(current, "data", "raw")
output_path = os.path.join(current, "data", "processed")
os.makedirs(output_path, exist_ok=True)

pdf_files = glob.glob(os.path.join(data_path, "*.pdf"))

records = []
chunker = Chunker(max_tokens=500, overlap=80)

# --- Process each PDF ---
for pdf_file in pdf_files:
    print(f"Processing: {pdf_file}")

    parser = PDFParser(pdf_file)
    parsed_pages = parser.extract()

    chunks = chunker.chunk(parsed_pages)

    # Add source-level metadata
    for chunk in chunks:
        chunk["metadata"]["source"] = {
            "file_name": os.path.basename(pdf_file)
        }

    records.extend(chunks)

# --- Save as JSONL (recommended for RAG) ---
jsonl_path = os.path.join(output_path, "chunks.jsonl")

with open(jsonl_path, "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved JSONL to: {jsonl_path}")


# --- Optional: Convert to DataFrame (for inspection/debugging) ---
df = pd.DataFrame(records)

# Flatten metadata for easier viewing
metadata_df = pd.json_normalize(df["metadata"])
df = pd.concat([df.drop(columns=["metadata"]), metadata_df], axis=1)

csv_path = os.path.join(output_path, "chunks_preview.csv")
df.to_csv(csv_path, index=False)

print(f"Saved preview CSV to: {csv_path}")


Processing: C:\Users\abeku\OneDrive\Desktop\Studybuddy\StudyBuddy-AI\data\raw\iesc102.pdf
Saved JSONL to: C:\Users\abeku\OneDrive\Desktop\Studybuddy\StudyBuddy-AI\data\processed\chunks.jsonl
Saved preview CSV to: C:\Users\abeku\OneDrive\Desktop\Studybuddy\StudyBuddy-AI\data\processed\chunks_preview.csv
